# STL Decomposition + Residual Thresholding

In [ ]:

import sys, time, warnings, os
from pathlib import Path

def find_root(marker='Data/train.csv'):
    for d in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (d / marker).exists():
            return d
    raise FileNotFoundError(f'Không thấy repo root (marker {marker}) từ {Path.cwd()}')

ROOT = find_root()
sys.path.append(str(ROOT / 'Modeling' / 'Code'))
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from statsmodels.tsa.seasonal import STL
from eval_protocol import time_split_per_kpi, evaluate_protocol
from preprocess import preprocess_all

df = pd.read_csv(ROOT / 'Data' / 'train.csv')
df.columns = ['timestamp', 'value', 'label', 'kpi']

ROBUST  = False    
MAX_GAP = 5
print('Repo root:', ROOT)
print('Số KPI trong train.csv:', df.kpi.nunique())

Số KPI trong train.csv: 26


In [2]:
def run_kpi_stl(gk):
    step = int(pd.Series(np.diff(np.sort(gk.timestamp.values))).mode().iloc[0])
    gk = time_split_per_kpi(gk, 0.6, 0.2)
    p = (preprocess_all(gk, max_gap_points=MAX_GAP, norm_method='robust')
         .sort_values('timestamp').reset_index(drop=True))
    y = p.label.values.astype(int); spl = p.split.values
    if y[spl == 'val'].sum() == 0 or y[spl == 'test'].sum() == 0:
        return None                                        # skip KPI thiếu nhãn ở val/test
    period = int(round(86400 / step))                      # 1440 cho 60s, 288 cho 300s
    min_len = 2 * period                                   # STL cần >= 2 chu kỳ
    J = max(period // 15, 1)                               # trend/low_pass jump → tăng tốc ~200x, AP không đổi

    real = p[p.value_filled.notna()]
    seg_len = real.groupby('segment').size()
    seg_ok = seg_len[seg_len >= min_len].index            # segment đủ dài cho STL
    p['stl_resid'] = np.nan
    for sid in seg_ok:                                     # phân rã từng segment (không bắc cầu qua gap)
        i = p.index[(p.segment == sid) & p.value_filled.notna()]
        r = STL(p.loc[i, 'value_norm'].values, period=period, robust=ROBUST,
                seasonal_jump=1, trend_jump=J, low_pass_jump=J).fit()
        p.loc[i, 'stl_resid'] = r.resid

    got = p.stl_resid.notna().values
    if got.sum() == 0:
        return None
    rtr = p.loc[(spl == 'train') & got, 'stl_resid']       # center/scale robust từ residual TRAIN
    med = rtr.median(); mad = np.median(np.abs(rtr - med)) * 1.4826
    mad = mad if mad > 1e-9 else 1e-9
    score = np.abs(p.stl_resid.values - med) / mad
    vm = (spl == 'val') & got; tm = (spl == 'test') & got
    if y[vm].sum() == 0 or y[tm].sum() == 0:
        return None                                        # STL bỏ hết anomaly val/test (segment ngắn)
    r = evaluate_protocol(y[vm], score[vm], y[tm], score[tm], step_s=step)
    return dict(kpi=gk.kpi.iloc[0][:8], n_anom_test=int(y[tm].sum()), period=period,
                n_seg_stl=len(seg_ok), cov_test=round(float(tm.sum() / max((spl == 'test').sum(), 1)), 3),
                AP=round(r['AP_pw'], 3), PW_F1=round(r['PW']['fbeta'], 3),
                PA_F1=round(r['PA']['fbeta'], 3), thr_pw=round(r['PW']['threshold'], 4),
                ttd_s=r['TTD']['ttd_sec_mean'])

In [3]:
rows, skipped = [], []
t0 = time.time()
for kpi, gk in df.groupby('kpi'):
    try:
        r = run_kpi_stl(gk.copy())
    except Exception as e:
        r = None
        print('  lỗi', kpi[:8], type(e).__name__)
    if r is None:
        skipped.append(kpi[:8])
    else:
        rows.append(r)
        print('done', r['kpi'], '| AP', r['AP'], '| PW_F1', r['PW_F1'],
              '| PA_F1', r['PA_F1'], '| seg', r['n_seg_stl'], 'cov', r['cov_test'])
print()
print('Skipped (' + str(len(skipped)) + '):', skipped)
print('Tổng thời gian: ' + str(round(time.time() - t0)) + 's')

done 02e99bd4 | AP 0.468 | PW_F1 0.434 | PA_F1 0.827 | seg 12 cov 0.94


done 07927a9a | AP 0.007 | PW_F1 0.0 | PA_F1 0.0 | seg 3 cov 1.0


done 09513ae3 | AP 0.003 | PW_F1 0.0 | PA_F1 0.0 | seg 10 cov 0.999


done 18fbb1d5 | AP 0.167 | PW_F1 0.004 | PA_F1 0.269 | seg 8 cov 0.999


done 1c35dbf5 | AP 0.282 | PW_F1 0.334 | PA_F1 0.0 | seg 10 cov 0.999


...

done affb01ca | AP 0.383 | PW_F1 0.412 | PA_F1 0.316 | seg 3 cov 1.0


done c58bfcba | AP 0.002 | PW_F1 0.0 | PA_F1 0.0 | seg 13 cov 0.999


done cff6d3c0 | AP 0.088 | PW_F1 0.154 | PA_F1 0.243 | seg 5 cov 0.997


done da403e4e | AP 0.69 | PW_F1 0.109 | PA_F1 0.886 | seg 8 cov 1.0


done e0770391 | AP 0.151 | PW_F1 0.044 | PA_F1 0.083 | seg 4 cov 0.909

Skipped (8): ['046ec29d', '54e8a140', '769894ba', '76f4550c', '8a20c229', '9bd90500', 'a5bf5d65', 'b3b2e6d1']
Tổng thời gian: 38s


## Bảng kết quả per-KPI + macro

In [4]:
res = pd.DataFrame(rows).sort_values('AP', ascending=False).reset_index(drop=True)
n_all = df.kpi.nunique()
mAP = res['AP'].mean(); mPW = res['PW_F1'].mean(); mPA = res['PA_F1'].mean(); mcov = res['cov_test'].mean()
print(f'MACRO AP    = {mAP:.3f}  (trên {len(res)}/{n_all} KPI)')
print(f'MACRO PW_F1 = {mPW:.3f} | MACRO PA_F1 = {mPA:.3f}')
print(f'Coverage test trung bình = {mcov:.3f} (phần điểm test được STL chấm)')
res

MACRO AP    = 0.280  (trên 18/26 KPI)
MACRO PW_F1 = 0.227 | MACRO PA_F1 = 0.325
Coverage test trung bình = 0.981 (phần điểm test được STL chấm)


,kpi,n_anom_test,period,n_seg_stl,cov_test,AP,PW_F1,PA_F1,thr_pw,ttd_s
0,9ee58794,761,1440,1,1.000,0.755,0.470,0.492,4.9801,150.000000
1,da403e4e,238,1440,8,1.000,0.690,0.109,0.886,1.6860,7.500000
2,7c189dd3,63,1440,3,1.000,0.502,0.587,0.333,9.5570,73.846154
3,02e99bd4,1282,1440,12,0.940,0.468,0.434,0.827,5.8406,37.500000
4,a40b1df8,83,1440,3,0.908,0.433,0.466,0.551,8.2658,67.500000
5,affb01ca,76,1440,3,1.000,0.383,0.412,0.316,11.5120,95.000000
6,8bef9af9,73,1440,3,1.000,0.377,0.366,0.751,10.3456,93.333333
7,88cf3a77,183,1440,1,1.000,0.290,0.254,0.615,6.0984,720.000000
8,1c35dbf5,2709,1440,10,0.999,0.282,0.334,0.000,2.0807,108.000000
9,40e25005,112,1440,2,1.000,0.167,0.254,0.153,5.8205,37.500000


In [ ]:
arima_path = ROOT / 'Modeling' / 'Stats' / 'ARIMA' / 'arima_per_kpi_config.json'
if_path = ROOT / 'Modeling' / 'ML' / 'IsolationForest' / 'if_per_kpi_config.json'
cmp = res[['kpi', 'AP', 'PW_F1']].rename(columns={'AP': 'STL_AP', 'PW_F1': 'STL_PW_F1'})
if os.path.exists(arima_path):
    a = pd.read_json(arima_path)[['kpi', 'AP']].rename(columns={'AP': 'ARIMA_AP'})
    cmp = cmp.merge(a, on='kpi', how='left')
if os.path.exists(if_path):
    f = pd.read_json(if_path)[['kpi', 'b_AP']].rename(columns={'b_AP': 'IF_feat_AP'})
    cmp = cmp.merge(f, on='kpi', how='left')
cols = [c for c in ['STL_AP', 'ARIMA_AP', 'IF_feat_AP'] if c in cmp.columns]
print('MACRO AP:', {c.replace('_AP', ''): round(float(cmp[c].mean()), 3) for c in cols})
if 'ARIMA_AP' in cmp.columns:
    nwin = int((cmp['STL_AP'] > cmp['ARIMA_AP']).sum())
    print(f'STL thắng ARIMA (theo AP): {nwin}/{len(cmp)} KPI')
cmp.sort_values('STL_AP', ascending=False).reset_index(drop=True)

MACRO AP: {'STL': 0.28, 'ARIMA': 0.381}
STL thắng ARIMA (theo AP): 1/18 KPI


,kpi,STL_AP,STL_PW_F1,ARIMA_AP
0,9ee58794,0.755,0.470,0.855
1,da403e4e,0.690,0.109,0.752
2,7c189dd3,0.502,0.587,0.699
3,02e99bd4,0.468,0.434,0.473
4,a40b1df8,0.433,0.466,0.722
5,affb01ca,0.383,0.412,0.615
6,8bef9af9,0.377,0.366,0.554
7,88cf3a77,0.290,0.254,0.242
8,1c35dbf5,0.282,0.334,0.292
9,40e25005,0.167,0.254,0.234


## Lưu config per-KPI (để inference dùng lại)

In [ ]:
out_path = ROOT / 'Modeling' / 'Stats' / 'STL' / 'stl_per_kpi_config.json'
res.to_json(out_path, orient='records', indent=1)
print('Đã lưu config per-KPI:', out_path)
print('Mỗi KPI có period, threshold PW, coverage và metrics riêng → dùng lại khi inference.')

Đã lưu config per-KPI: C:\Projects\anomaly-detection-fundamentals\Modeling\Stats\STL\stl_per_kpi_config.json
Mỗi KPI có period, threshold PW, coverage và metrics riêng → dùng lại khi inference.
